In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import make_column_transformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_PATH = os.path.join('..', 'vr_legibility_train.csv')

In [2]:
# 1. 데이터 로드 및 전처리
df = pd.read_csv(DATA_PATH)[["Yaw", "Pitch", "Size", "Label"]]

# 특성(X)과 타겟(y) 분리
X = df.drop('Label', axis=1)
y = df['Label']

# 2. 학습/테스트 데이터 분할 (Stratified Sampling)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# 학습/테스트 데이터 분할 (StratifiedKFold 적용)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
# 2. 파이프라인 (스케일링 + 모델) 구축
# ---------------------------------------------------------
preprocessor = make_column_transformer(
    (MinMaxScaler(), ['Yaw', 'Pitch']),
    (make_pipeline(FunctionTransformer(np.log1p), MinMaxScaler()), ['Size'])
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('svm', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

In [ ]:
#그리드 서치 하이퍼파라미터 범위
param_grid = [
    {
        'penalty': ['l2'], 
        'C': [0.001, 0.01, 0.1, 1, 10, 100], 
        'solver': ['lbfgs'] # l2에 가장 무난한 기본 알고리즘
    },
    {
        'penalty': ['l1'], 
        'C': [0.001, 0.01, 0.1, 1, 10, 100], 
        'solver': ['liblinear'] # l1을 쓰려면 solver를 바꿔줘야 에러가 안 납니다
    }
]

In [ ]:
grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', # 모델 평가 기준 (만약 데이터 불균형이 심하면 'f1'으로 변경)
    n_jobs=-1           # 컴퓨터의 모든 CPU 코어를 사용하여 연산 속도 최대화
)